# Typed findings — and an agent that builds other agents

Day 1 returned `reply.body` as a markdown string. Today an agent returns a Pydantic object — and a planner agent emits a typed `ResearchPlan` whose `team` field becomes the workers we spawn at runtime. We run them in pipeline order — researcher → analyst → critic — sharing one `MemoryStream` so each later worker actually sees what the researcher found instead of inventing sources. Only the researcher needs a typed `Finding`; the analyst and critic respond in markdown, since their job is to read what's already in the conversation. While the researcher runs, an `@agent.observer(...)` records every search-result `title → url` pair, so we can render every citation with the real source link.

Make sure `OPENAI_API_KEY` and `EXA_API_KEY` are in your `.env`.

In [1]:
import os
from typing import Literal

from dotenv import load_dotenv
from IPython.display import Markdown, display
from pydantic import BaseModel, Field

from autogen.beta import Agent, MemoryStream
from autogen.beta.config import OpenAIConfig
from autogen.beta.events import ToolResultsEvent
from autogen.beta.tools import ExaToolkit

load_dotenv()

config = OpenAIConfig(
    model="gpt-5.4-mini",
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://api.openai.com/v1",
)
exa = ExaToolkit(api_key=os.getenv("EXA_API_KEY"))

## The schemas

Field descriptions are not decoration — they go into the JSON schema the model sees, so they double as instructions. Heads-up on `strict` mode: every field has to be required, so no Pydantic defaults below. (Need defaults? Wrap with `PromptedSchema(...)` to skip strict mode.)

In [2]:
class Citation(BaseModel):
    title: str = Field(description="Exact title of a search result you're citing — must match a hit returned by your tools.")
    url: str = Field(description="URL of that same search result.")
    relevance: str = Field(description="One sentence on why this source supports your claim.")

class Finding(BaseModel):
    """What the researcher returns."""
    topic: str = Field(description="The specific aspect of the research topic this finding addresses.")
    summary: str = Field(description="2-3 sentences capturing the core finding.")
    citations: list[Citation] = Field(description="Cite only sources that came back from your search tool — never invent. Emit [] if you didn't search.")
    novelty: float = Field(ge=0.0, le=1.0, description="0 = well-known, 1 = cutting edge.")

WorkerRole = Literal["researcher", "analyst", "critic"]

class WorkerSpec(BaseModel):
    role: WorkerRole = Field(description="What kind of work this worker does.")
    name: str = Field(description="Short lowercase identifier, no spaces.")
    prompt: str = Field(description="System prompt for this worker, focused on its role.")
    tools: list[str] = Field(description="Tool keys. Use 'exa_search' for researchers; emit [] for none.")

class ResearchPlan(BaseModel):
    topic: str = Field(description="The research topic.")
    description: str = Field(description="Why this topic is worth investigating.")
    team: list[WorkerSpec] = Field(description="Pipeline order: researcher first (to gather sources), then analyst, then critic.")

## A planner agent

An agent with `response_schema=ResearchPlan` returns a typed plan describing the workers it wants spun up, in pipeline order.

In [3]:
planner = Agent(
    name="planner",
    prompt=(
        "You design tight 3-person research pipelines: researcher → analyst → critic, in that order. "
        "The researcher gets exa_search; analyst and critic need no tools and reason over the upstream findings."
    ),
    config=config,
    response_schema=ResearchPlan,
)

plan: ResearchPlan = await (await planner.ask(
    "Plan a small team to investigate high-Tc superconductivity."
)).content()

plan_lines = [f"**{plan.topic}** — {plan.description}", ""]
for w in plan.team:
    plan_lines.append(f"**[{w.role}] {w.name}** — tools={w.tools or 'none'}")
    plan_lines.append(f"> {w.prompt}")
    plan_lines.append("")
display(Markdown("\n".join(plan_lines)))

**High-Tc superconductivity** — High-Tc superconductivity remains one of the central open problems in condensed matter physics, with implications for quantum materials, lossless power transmission, and future electronic devices. A small researcher→analyst→critic pipeline can efficiently gather current literature, synthesize competing theoretical perspectives, and stress-test conclusions for gaps or overclaims.

**[researcher] researcher** — tools=['exa_search']
> You are the researcher. Use exa_search to gather a concise set of high-quality sources on high-Tc superconductivity, prioritizing recent reviews, landmark experimental papers, and key theoretical overviews. Focus on cuprates, iron-based superconductors, and any consensus or controversy about pairing mechanisms. Return source titles, authors, years, and one-line relevance notes.

**[analyst] analyst** — tools=none
> You are the analyst. Synthesize the researcher’s findings into a structured overview: major material families, leading pairing theories, experimental signatures, open questions, and points of agreement vs. disagreement. Identify patterns across sources and highlight the most credible claims.

**[critic] critic** — tools=none
> You are the critic. Evaluate the analyst’s synthesis for overconfidence, missing alternative hypotheses, weak evidence, and conflation of correlation with causation. Flag any claims that need stronger support and suggest what additional evidence would improve confidence.


## Run the pipeline on a shared stream

All three workers share one `MemoryStream`. The researcher returns a typed `Finding`; the analyst and critic read that Finding plus the researcher's search history off the shared stream, and respond in markdown — they don't need a schema. The `@worker.observer(ToolResultsEvent)` records every search-result `title → url` pair into a shared dict, and we case-fold both sides at lookup time so capitalization or whitespace drift doesn't break a match.

In [4]:
TOOLBOX = {"exa_search": [exa.search()], "exa_answer": [exa.answer()]}

shared_stream = MemoryStream()
title_to_url: dict[str, str] = {}

def norm(s: str) -> str:
    return s.lower().strip()

async def run_worker(spec: WorkerSpec, topic: str) -> Finding | str:
    typed = spec.role == "researcher"
    schema_note = (
        "Every Citation.title must be the exact title of a real search result that appears in this conversation — never invent sources."
        if typed else
        "Read what the prior workers said in the conversation and write a concise markdown response."
    )
    tools = [t for key in spec.tools for t in TOOLBOX.get(key, [])]
    worker = Agent(
        name=spec.name,
        prompt=spec.prompt,
        config=config,
        tools=tools,
        response_schema=Finding if typed else None,
    )

    @worker.observer(ToolResultsEvent)
    def _collect(event: ToolResultsEvent) -> None:
        for r in event.results:
            data = r.result.parts[0].data
            for hit in getattr(data, "results", None) or []:
                title, url = getattr(hit, "title", None), getattr(hit, "url", None)
                if title and url:
                    title_to_url[title] = url

    reply = await worker.ask(
        f"Topic: {topic}\n\nContribute as {spec.role}. {schema_note}. Respond concisely",
        stream=shared_stream,
    )
    return await reply.content() if typed else (reply.body or "")

results: list[Finding | str] = []
for spec in plan.team:
    results.append(await run_worker(spec, plan.topic))

## Render — every link comes from the source

Match each `Citation.title` against the dict the observer filled, case- and whitespace-insensitive. A match (✓) renders a hyperlink built from the real source URL; a miss (✗) shows the title alone.

In [5]:
norm_to_real = {norm(t): (t, u) for t, u in title_to_url.items()}

lines: list[str] = []
for spec, result in zip(plan.team, results):
    lines.append(f"### {spec.name} ({spec.role})")
    if isinstance(result, Finding):
        lines.append(f"*{result.topic} — novelty={result.novelty:.2f}*")
        lines.append(result.summary)
        for c in result.citations:
            match = norm_to_real.get(norm(c.title))
            if match:
                real_title, url = match
                lines.append(f"- ✓ [{real_title}]({url}) — {c.relevance}")
            else:
                lines.append(f"- ✗ {c.title} — {c.relevance}")
    else:
        lines.append(result)
    lines.append("")

display(Markdown("\n".join(lines)))

### researcher (researcher)
*High-Tc superconductivity: cuprates vs iron-based materials, pairing symmetry, and current consensus/controversies — novelty=0.00*
Recent reviews agree that cuprates are widely believed to have predominantly d-wave pairing, while iron-based superconductors are broadly described by sign-changing s± pairing, though nodal and material-dependent variants remain active areas of debate. The pairing glue is still controversial in both families, but spin fluctuations are the leading consensus framework; newer work emphasizes charge correlations in cuprates and orbital-selective, Hund’s-metal physics in iron-based systems.
- ✗ Charge Correlations in Cuprate Superconductors — 2024 review of cuprate charge order and its debated relation to superconductivity/pairing.
- ✗ Iron pnictides and chalcogenides: a new paradigm for superconductivity — Major 2022 review summarizing iron-based superconductors, pairing symmetry, nematicity, magnetism, and open mechanism questions.
- ✗ Iron-based superconductors: teenage, complex, challenging — 2023 status update on iron-based superconductors, highlighting correlations, orbital selectivity, and unresolved pairing issues.
- ✗ Ironing out the details of unconventional superconductivity — Comprehensive 2022 overview of iron-based superconductivity, including dominant pairing scenarios and open questions.
- ✓ [The electron-pairing mechanism of iron-based superconductors](https://pubmed.ncbi.nlm.nih.gov/21474751/) — Classic theoretical review framing the pairing-mechanism debate in iron-based superconductors.
- ✗ Experimental determination of the superconducting pairing state in YBCO from the phase coherence of YBCO-Pb dc SQUIDs — Landmark phase-sensitive cuprate experiment giving strong evidence for d-wave pairing.
- ✗ Phase-sensitive tests of the symmetry of the pairing state in the high-temperature superconductors---Evidence for ${d}_{{x}^{2}\ensuremath{-}{y}^{2}}$ symmetry — Foundational review synthesizing phase-sensitive evidence for d-wave pairing in cuprates.
- ✗ Angle-resolved photoemission studies of the cuprate superconductors — Key cuprate ARPES review on the gap, pseudogap, and electronic structure relevant to pairing debates.
- ✗ Nodal s± pairing symmetry in an iron-based superconductor with only hole pockets — 2024 experiment arguing for nodal s± symmetry in KFe2As2, important for the iron-based pairing debate.
- ✓ [Pocket pairs in iron-based materials](https://www.nature.com/articles/s41567-023-02375-y?error=cookies_not_supported&code=b9f4e218-61f3-4e5e-985d-4cf7601a1856) — 2024 commentary highlighting new high-resolution evidence that may unify pairing symmetry in hole-doped iron-based superconductors.

### analyst (analyst)
## High-Tc superconductivity: concise synthesis

### Major material families
- **Cuprates**: still the clearest high-Tc platform; consensus remains strongest here.
- **Iron-based superconductors (FeSCs)**: pnictides and chalcogenides; more structurally and electronically diverse, but highly informative for unconventional pairing.

### Leading pairing theories
- **Cuprates**: overwhelmingly support **predominantly d-wave** pairing.
- **FeSCs**: leading framework is **sign-changing s± pairing**, often tied to **spin fluctuations**; however, **nodal s±**, **d-wave competition**, and in some contexts **orbital-selective / Hund’s-metal** perspectives remain active.
- Across both families, the most credible common theme is **correlation-driven, non-phononic pairing**, with **spin fluctuations** the most widely supported glue candidate.

### Experimental signatures
- **Cuprates**
  - Phase-sensitive Josephson/SQUID experiments gave the strongest evidence for **d-wave symmetry**.
  - ARPES and neutron scattering broadly support anisotropic gap structure and sign-changing behavior.
  - Recent emphasis: **charge-density-wave/charge correlations** as a major competing or intertwined order.
- **FeSCs**
  - ARPES, neutron spin resonance, penetration depth, thermal conductivity, and disorder studies probe gap symmetry.
  - New 2024 work on **KFe2As2** strengthens the case for **nodal s±** or closely related sign-changing states.
  - Strong evidence for **orbital selectivity**, **nematicity**, and **Hund’s-metal behavior** in the normal state.

### Open questions
- What is the **true microscopic pairing glue** in each family?
- How do **charge order, nematicity, magnetism, and superconductivity** interlock in cuprates?
- In FeSCs, what determines the crossover among **s±, nodal s±, and d-wave** states?
- How much of superconductivity is controlled by **orbital-selective correlations** versus itinerant spin fluctuations?
- Can one obtain a **unified theory** across cuprates and FeSCs, or are they only analogous at the phenomenological level?

### Agreement vs disagreement
**Broad agreement**
- High-Tc superconductivity is **unconventional** and strongly linked to **electron correlations**.
- **Phonons alone** are not viewed as the dominant explanation in these systems.
- Gap symmetry is **highly diagnostic** and material-dependent.

**Main disagreements**
- Cuprates: far less disagreement on symmetry, but ongoing debate on **pairing mechanism** and the role of **charge order**.
- FeSCs: substantial debate persists on **pairing symmetry and glue**, especially in hole-only and strongly correlated regimes.

### Most credible claims
- **Cuprate d-wave pairing** is the most secure conclusion.
- **FeSC s± spin-fluctuation pairing** is the leading framework, but with real material-dependent exceptions and nodal variants.
- The strongest modern trend is toward a **multifactor picture**: pairing emerges from an interplay of **spin, orbital, charge, and lattice effects**, not a single simple mechanism.

### critic (critic)
## Critique of prior synthesis

- **Overconfidence in consensus**
  - Saying cuprate **d-wave** pairing is “overwhelmingly” settled is broadly fair, but the mechanism is still not settled. The response sometimes implies more closure than exists.
  - For Fe-based superconductors, calling **s±** the “leading framework” is plausible, but the field is still fragmented; this should be framed as a dominant hypothesis, not near-consensus.

- **Missing alternative hypotheses**
  - The summary underplays competing ideas in cuprates, e.g. roles for **charge order**, **pairing fluctuations**, and proposals beyond simple spin-fluctuation pairing.
  - For FeSCs, it should mention explicit alternatives such as **d-wave**, **orbital-fluctuation**, and more exotic **triplet / orbital-selective** proposals.

- **Conflation of correlation with causation**
  - The phrase “correlation-driven, non-phononic pairing” is too strong as a causal statement. Strong correlations are clearly involved, but direct proof that they *cause* pairing is incomplete.
  - Likewise, stating spin fluctuations are the “most widely supported glue candidate” is reasonable, but it is still inference from multiple probes, not a direct demonstration.

- **Weak evidence / unsupported generalization**
  - “The strongest modern trend is toward a multifactor picture” is a defensible review-level statement, but it needs caveats: this is a synthesis of disparate observations, not a single established theory.
  - “Recent 2024 work on KFe2As2 strengthens the case for nodal s±” should be tempered: it supports that interpretation in that material, not universally across FeSCs.

### What would improve confidence
- Direct, material-specific comparisons across **phase-sensitive**, **ARPES**, **neutron**, **thermal transport**, and **disorder** experiments.
- Clear separation between evidence for **gap symmetry** and evidence for **pairing mechanism**.
- More explicit discussion of where data are **consistent with multiple models** rather than uniquely identifying one.


## Up next

What worked: a typed handoff (planner → `ResearchPlan` → workers), a researcher → analyst → critic pipeline sharing one `MemoryStream` so later workers see real sources instead of inventing, an `@worker.observer(ToolResultsEvent)` that captured `title → url` live, and a tolerant case-folded match so the render uses the canonical title and URL straight from the search hit. Findings → text → text — only the first stop needs a schema. No group-chat coordinator either; agents compose via typed objects + a shared stream + observers in beta. (If you really need v0's `GroupChat`, bridge with `agent.as_conversable()`.)

Day 3: zoom in on the simplest two-worker form — Surveyor → Theorist.